# A test for each type of tool (chapter 5)

Four templates, one per type: seeded packs for vision, coded stops for downtime, a question set for a copilot and a held out period for a forecast. Every file is synthetic. Replace each with your own records.

In [ ]:
import pandas as pd

## Vision: seeded packs and false rejects (5.1)

Seed 200 known bad packs through the line and count the catches. Then price the false rejects at line speed.

In [ ]:
packs = pd.read_csv("../data/ch05_seeded_packs.csv")
print(packs.groupby("defect")["rejected"].agg(["count", "mean"]).round(2))
print("Caught overall:", round(packs["rejected"].mean(), 3))

bottles_an_hour, false_reject_rate = 36_000, 0.005
print("Bottles a second:", bottles_an_hour / 3600)
print("Good bottles binned an hour:", bottles_an_hour * false_reject_rate)

## Downtime: logged stops against coded stops (5.2)

The line control system records every stop. Operators log the big ones.

In [ ]:
stops = pd.read_csv("../data/ch05_filler_stops.csv")
all_h = stops["duration_s"].sum() / 3600
logged_h = stops.loc[stops["logged_by_operator"] == 1, "duration_s"].sum() / 3600
print(f"Logged by operators: {logged_h:.0f} h; line control system: {all_h:.0f} h")
print(f"A 25 percent cut is {0.25 * logged_h:.1f} h or {0.25 * all_h:.1f} h; the baselines differ by {all_h - logged_h:.0f} h")
print("Stops under 30 s:", (stops["duration_s"] <= 30).sum())

## Copilot: a fifty question test set (5.3)

The shift leader checks every answer. Hours saved count only if a job changes.

In [ ]:
qs = pd.read_csv("../data/ch05_copilot_test_set.csv")
print("Questions:", len(qs), "| correct:", qs["correct"].mean().round(2), "| cite a document:", qs["cites_document"].mean().round(2))
print(qs.groupby("category")["correct"].mean().round(2))

hours_a_week, checking, rate = 10, 3, 40
print("Hours a year:", hours_a_week * 52, "| USD:", hours_a_week * 52 * rate, "| net hours a week:", hours_a_week - checking)

## Forecast: beat the naive forecast on a held out period (5.4)

In [ ]:
sales = pd.read_csv("../data/ch05_weekly_sales.csv")
held = sales[sales["period"] == "held_out"]

def mape(forecast):
    return (100 * (held["actual_hL"] - forecast).abs() / held["actual_hL"]).mean()

naive, vendor = mape(held["naive_same_week_last_year"]), mape(held["vendor_forecast"])
print(f"Naive error {naive:.1f} percent; vendor error {vendor:.1f} percent")
print(f"Relative gain {100 * (naive - vendor) / naive:.0f} percent; in points {naive - vendor:.1f}")
print("Book example: 30 x (1 - 0.20) =", 30 * (1 - 0.20))